### 1. Install Kaggle and Dependencies

In [1]:
#This is done only the first time.

# !pip install kaggle

Mount the Google Drive for direct access to files (Kaggle API)

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Make a Temporary Directory for Kaggle to Import Files

In [3]:
# Remove any existing Temporary Directory ".kaggle/"

# ! rm -r ~/.kaggle

In [4]:
# This creates a new Temporary Directory ".kaggle/"

! mkdir ~/.kaggle


Import the Kaggle API to the Temporary Location (.kaggle/)

In [5]:
! cp "/content/drive/MyDrive/Colab Notebooks/KaggleAPI/kaggle.json" ~/.kaggle/

cp: cannot stat '/content/drive/MyDrive/Colab Notebooks/KaggleAPI/kaggle.json': No such file or directory


Download Dataset from Kaggle

In [6]:
!kaggle datasets download kmader/skin-cancer-mnist-ham10000

Dataset URL: https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000
License(s): CC-BY-NC-SA-4.0
100% 5.20G/5.20G [02:14<00:00, 41.5MB/s]



### 2. Import Libraries

In [7]:
# Linear Algebra

import numpy as np

# Data Processing

import pandas as pd

**for dirname, _, filenames in os.walk('/kaggle/input'):**

This line uses the os.walk() function to traverse a directory tree.
It starts from the directory /kaggle/input. os.walk() yields three values for each directory it visits: the directory path (dirname), a list of subdirectories (_, which we are ignoring here), and a list of filenames (filenames) within that directory.

**for filename in filenames:**

This nested loop iterates through each filename in the list of filenames obtained from os.walk().

**print(os.path.join(dirname, filename)):**

This line constructs the full path to each file by joining the directory name (dirname) and the filename (filename) using os.path.join().

The full path of each file is then printed to the console.


In [8]:
# Interact with the Pyhton OS; File and Directory Operations.

import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [9]:
# Data Visualisation

import matplotlib.pyplot as plt
import seaborn as sns

# Training and Evaluating the Model

from sklearn.model_selection import train_test_split
from sklearn.utils import class_weight
from sklearn.metrics import accuracy_score, classification_report

# TensorFlow for Deep Learning

import tensorflow as tf

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2

from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout

### 3. Prepare the Dataset

Zipfile Module provides the capabilities to perform operations on **.zip** files.

In [10]:
import zipfile

# Unzip the Downloaded Dataset

with zipfile.ZipFile('skin-cancer-mnist-ham10000.zip', 'r') as zip_ref:
    zip_ref.extractall('.')

Shutil Module provides the capabilities to perform operations on a set of high-level operations on files and collections of files, often referred to as "shell utilities".

In [11]:
import shutil

# Define the Paths to Source Directories

directory1 = '/content/HAM10000_images_part_1'
directory2 = '/content/HAM10000_images_part_2'

# Define the Path for the New Destination Directory

destination_directory = '/content/HAM10000_images'

# Create the Destination Directory

if not os.path.exists(destination_directory):
    os.makedirs(destination_directory)

# Function to Copy Files from Source directories to Destination Directory

def copy_files(source_dir, dest_dir):
    for filename in os.listdir(source_dir):
        source_path = os.path.join(source_dir, filename)
        destination_path = os.path.join(dest_dir, filename)
        if os.path.isfile(source_path):
            shutil.copy2(source_path, destination_path)

# Copy files from the First directory

copy_files(directory1, destination_directory)

# Copy files from the second directory

copy_files(directory2, destination_directory)

print("Directories have been Merged.")

Directories have been Merged.


Loading the Metadata

In [12]:
# Load the Metadata

metadata = pd.read_csv('HAM10000_metadata.csv')
metadata.head()

,lesion_id,image_id,dx,dx_type,age,sex,localization
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear


Load the Image Path

In [13]:
# Load the Image Path

image_dir = './HAM10000_images'

# Adding the Image Path to the Corressponding Values

metadata['image_path'] = metadata['image_id'].apply(lambda x: os.path.join(image_dir, f'{x}.jpg'))

metadata.head()

,lesion_id,image_id,dx,dx_type,age,sex,localization,image_path
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp,./HAM10000_images/ISIC_0027419.jpg
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp,./HAM10000_images/ISIC_0025030.jpg
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp,./HAM10000_images/ISIC_0026769.jpg
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp,./HAM10000_images/ISIC_0025661.jpg
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear,./HAM10000_images/ISIC_0031633.jpg


### 4. Transform the Image Data and Generate Training Validation Sets

We use different augmentation techniques help in increasing the size and diversity of the training dataset, which can improve the model's generalization ability and prevent it from overfitting to the training data.

These are -

1. **rescale=1./255**

This Normalizes the pixel values to scale pixel values to the range [0, 1], which helps in better model training.

2. **validation_split=0.2**

This reserves 20% of the data for validation during training.

3. **horizontal_flip=True**

This randomly flips images horizontally.

4. **vertical_flip=True**

This randomly flips images vertically.

5. **zoom_range=0.2**

This randomly zooms into images by up to 20%.

6. **rotation_range=20**

This randomly rotates images by up to 20 degrees.

In [14]:
# Create an Object of ImageDataGenerator

datagen = ImageDataGenerator(
    preprocessing_function = preprocess_input,
    validation_split = 0.2,
    horizontal_flip = True,
    vertical_flip = True,
    zoom_range = 0.2,
    shear_range = 0.2,
    rotation_range = 20
)

Generating Training and Validation Sets

In [15]:
# Defining the Target Image Size

target_size = (224, 224) # Resize Images to 224x224 — MobileNetV2 performs best at this input size.


# Batch Size is the number of Images processed at Once. This can be changed depending on the System Memory.

batch_size = 32

In [16]:
# Generating the Training Data

train_generator = datagen.flow_from_dataframe(
    dataframe = metadata,
    x_col = 'image_path',      # Feature Column (Image Data)
    y_col = 'dx',              # Target Column (Skin Cancer Classification)
    subset = "training",
    batch_size = batch_size,
    seed = 42,
    shuffle = True,
    class_mode = 'categorical',
    target_size = target_size
)

Found 8012 validated image filenames belonging to 7 classes.


In [17]:
# Generating the Validation Data

validation_generator = datagen.flow_from_dataframe(
    dataframe = metadata,
    x_col = 'image_path',      # Feature Column (Image Data)
    y_col = 'dx',              # Target Column (Skin Cancer Classification)
    subset = "validation",
    batch_size = batch_size,
    seed = 42,
    shuffle = True,
    class_mode = 'categorical',
    target_size = target_size
)

Found 2003 validated image filenames belonging to 7 classes.


### 5. Build and Train the Model (Transfer Learning)

Calculating Class Weights

In [18]:
# Calculating Class Weights to manage Class Imbalance

from sklearn.utils import class_weight

class_weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_generator.classes),
    y=train_generator.classes
)
class_weights = dict(enumerate(class_weights))

Loading the Base Model

In [19]:
# Load the Base Model without Top Layer

base_model = MobileNetV2(input_shape = (224, 224, 3), weights = 'imagenet', include_top = False)
base_model.trainable = False  # Freeze Base Layers

# Compile the Custom Head Model on top of the Base Model

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.2)(x)
output = Dense(len(train_generator.class_indices), activation='softmax')(x)

model_mobilenet = Model(inputs=base_model.input, outputs=output)

# Compile the Custom Head Model on top of the Base Model

model_mobilenet.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [ ]:
# Train the Model

model_mobilenet.fit(train_generator, validation_data=validation_generator, epochs=10, class_weight=class_weights)

Epoch 1/10
 86/251 ━━━━━━━━━━━━━━━━━━━━ 2:08 780ms/step - accuracy: 0.8080 - loss: 3.3428

Training the Model with Callbacks

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

callbacks = [
    EarlyStopping(patience=5, restore_best_weights=True),
    ReduceLROnPlateau(patience=2, factor=0.2, min_lr=1e-6),
    ModelCheckpoint('best_model.h5', save_best_only=True)
]

history = model_mobilenet.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=20,
    callbacks=callbacks
)


Predictions from the Model

In [ ]:
# Generating Predictions from the Model

y_true = validation_generator.classes
y_pred_proba = model_mobilenet.predict(validation_generator, verbose=1)
y_pred = np.argmax(y_pred_proba, axis=1)

Evaluating Performance of the Model

In [ ]:
 # Calculating Accuracy and Loss of the Model

val_loss, val_accuracy = model_mobilenet.evaluate(validation_generator, verbose=1)
print(f"\n✅ Validation Loss: {val_loss:.4f}")
print(f"✅ Validation Accuracy: {val_accuracy:.4f}")